# Portfolio Metrics — review notebook

Runs the pipeline in **replay mode** (from the committed cache, no API key) and renders the
Q2 2025 cross-section, the provenance trail, the reconciliation cross-check, and accuracy.

Run from the repo root: `jupyter lab review.ipynb`.

In [ ]:
import sys; sys.path.insert(0, '.')
from src import reconcile, table, evaluate
from src.llm_extract import extract_all
from src.pdf_text import list_pdfs

pdfs = [p for p in list_pdfs('data') if 'Portfolio_Snapshot' not in p.name]
records = extract_all(pdfs, force=False)   # replay from cache
recon = reconcile.load()
rows = table.build_long_rows(records, recon)
pivot = table.build_pivot(rows, recon)
print(f'{len(records)} reports -> {len(rows)} metric rows')

## Q2 2025 portfolio cross-section
Grouped by business model. `~` prose · `(ftn)` footnote · `*restated` · `--` not disclosed · `n/a` not applicable.

In [ ]:
def _style(v):
    v = str(v)
    if v == 'n/a':            return 'color:#cccccc'
    if v == '--':             return 'color:#e69138'
    if '*restated' in v:      return 'background-color:#fff2cc'
    if '(ftn)' in v or '~' in v: return 'color:#3d6'
    return ''

try:
    display(pivot.style.map(_style))
except AttributeError:
    display(pivot.style.applymap(_style))

## Provenance trail (example)
Every value carries the verbatim source quote + page it was extracted from.

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
cols = ['metric_display', 'value_raw', 'raw_label', 'source_type', 'match_type', 'provenance', 'page', 'source_quote']
df[(df.company_key == 'novacloud') & (df.period == 'Q2 2025')][cols].reset_index(drop=True)

## Reconciliation vs Sagard's manual snapshot

In [ ]:
import pandas as pd
pd.DataFrame(reconcile.cross_check(rows, recon))

In [ ]:
acc = evaluate.evaluate(rows)
print(f"accuracy vs hand-keyed ground truth: {acc['correct']}/{acc['total']} = {acc['accuracy']}%")
acc['misses'] or 'no misses'